# Inflation curves (CPI / breakeven and real rates)

Deep-dive reference: explicit CPI projection clocks and immutable monthly fixing snapshots.

## Concept

**Inflation curves** describe expected **CPI** growth or **inflation swap** breakevens, used to price **TIPS**, **ILBs**, and **inflation swaps**. A common identity (Fisher-style, continuous-time shorthand) links **nominal** \(r_n\), **real** \(r_r\), and **inflation** \(\pi\): \(r_n \approx r_r + \pi\). When no dedicated curve type exists, you can still illustrate **real zero rates** from a **nominal** curve and an assumed **inflation** path.

## API walkthrough

`InflationCurve` stores unlagged absolute CPI projections. Supply an origin knot, day count, interpolation and extrapolation explicitly. Query levels with `cpi(t)` and annualized growth with `inflation_rate(t1, t2)`. Contractual lags, reference-index interpolation and principal floors belong to instruments, not this source curve.

In [ ]:
from datetime import date
import sys
AS_OF = date(2025, 1, 15)
sys.path.insert(0, "../..")


from finstack_quant.core.market_data import InflationCurve

curve = InflationCurve(
    "USD-CPI",
    AS_OF,
    [(0.0, 315.5), (1.0, 323.4), (2.0, 331.8), (5.0, 357.9), (10.0, 405.4)],
    day_count="act_365f",
    interp="log_linear",
    extrapolation="extend",
)
print("curve:", curve)
print("base_cpi:", curve.base_cpi, "day_count:", curve.day_count)
for t in (1.0, 2.0, 5.0):
    print(f"t={t}y  cpi={curve.cpi(t):.3f}")
print("5Y annualized inflation:", round(curve.inflation_rate(0.0, 5.0), 6))


## Practical example

**Nominal OIS** discount curve plus a **constant CPI breakeven** (illustrative) to derive **approximate real zeros** with \(r_r \approx r_n - \pi\).

**Caveats:** Real **CPI** is **lagged** (release delay and contract **observation lags**), **seasonal** (and often revised), and market breakevens embed an **inflation risk premium** and liquidity effects—so a flat \(\pi\) is only a teaching shortcut. In **continuous compounding**, if nominal and real zeros are defined consistently, \(DF(t) \approx e^{-r t}\) links discounting to zeros; a Fisher-style split \(r_n \approx r_r + \pi\) is still an approximation and must use compatible conventions in production.

In [ ]:
# Unlagged source growth is not a contractual linker indexation ratio.
notional = 1_000_000.0
for t in (1.0, 3.0, 5.0, 10.0):
    ratio = curve.cpi(t) / curve.base_cpi
    adjusted_principal = notional * ratio
    print(
        f"t={t:>4.1f}y  source_growth_factor={ratio:.6f}  illustrative_notional={adjusted_principal:,.2f}"
    )

## Takeaways

- `InflationCurve` exposes absolute unlagged CPI levels and annualized growth on an explicit source clock.
- `InflationIndex` holds first-of-month observations with availability dates and a knowledge cutoff; expected releases are separate schedule facts.
- Instrument valuation resolves contractual lag, monthly reference-index interpolation and deflation protection. Source growth alone is not a linker uplift factor.
- Calibrated inflation curves still depend on discounting, seasonality choices, and instrument conventions, so treat the simple examples here as structure-first rather than full production setup.

## Calibration from inflation swap quotes

With the calibration engine, we can bootstrap an **inflation curve** from zero-coupon inflation swap (ZCIS) rates.  This requires:
1. A pre-existing **discount curve** (for PV calculations)
2. ZCIS quotes, an explicit contractual base CPI and lag, a source CPI origin/clock, and a fixing snapshot with release assumptions

The engine solves for the CPI projection path that reprices each ZCIS to zero NPV.

In [ ]:
from datetime import date
import json
from pathlib import Path

from finstack_quant.calibration import calibrate
from finstack_quant.core.market_data import InflationIndex, InflationObservation, InflationExpectedRelease
AS_OF = date(2025, 1, 15)

_NOTEBOOK_DATA = json.loads(Path("data/inflation_curves.json").read_text())
BASE_DATE = AS_OF.isoformat()


def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}


rate_quotes = [
    {"kind": "rate_quote", "type": "deposit", "id": "USD-SOFR-DEP-1M", "index": "USD-SOFR-OIS", "pillar": tenor(1, "months"), "rate": 0.0525},
    {"kind": "rate_quote", "type": "deposit", "id": "USD-SOFR-DEP-3M", "index": "USD-SOFR-OIS", "pillar": tenor(3, "months"), "rate": 0.0520},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-1Y", "index": "USD-SOFR-OIS", "pillar": tenor(1, "years"), "rate": 0.0510},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-2Y", "index": "USD-SOFR-OIS", "pillar": tenor(2, "years"), "rate": 0.0490},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-5Y", "index": "USD-SOFR-OIS", "pillar": tenor(5, "years"), "rate": 0.0450},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-10Y", "index": "USD-SOFR-OIS", "pillar": tenor(10, "years"), "rate": 0.0410},
]

inflation_quotes = _NOTEBOOK_DATA['inflation_quotes']
fixings = InflationIndex("USD-CPI", [InflationObservation("2024-10-01", 315.5, "2024-11-15"), InflationObservation("2024-11-01", 315.5, "2024-12-15"), InflationObservation("2024-12-01", 315.5, "2025-01-15")], "USD", AS_OF)
releases = [InflationExpectedRelease(date(y, m, 1), date(y + (m == 12), m % 12 + 1, 15), AS_OF) for y in range(2025, 2036) for m in range(1, 13)]
fixings = fixings.with_expected_releases(releases)

plan = {
    "schema": "finstack_quant.calibration/1",
    "plan": {
        "id": "us-cpi-curve",
        "description": "US CPI inflation curve from ZCIS quotes",
        "quote_sets": {
            "ois": [q["id"] for q in rate_quotes],
            "zcis": [q["inflation_swap"]["id"] for q in inflation_quotes],
        },
        "steps": [
            {
                "id": "USD-OIS",
                "quote_set": "ois",
                "kind": "discount",
                "curve_id": "USD-OIS",
                "currency": "USD",
                "base_date": BASE_DATE,
                "method": "bootstrap",
                "interpolation": "linear",
                "extrapolation": "extend",
            },
            {
                "id": "USD-CPI",
                "quote_set": "zcis",
                "kind": "inflation",
                "curve_id": "USD-CPI",
                "currency": "USD",
                "base_date": BASE_DATE,
                "discount_curve_id": "USD-OIS",
                "index": "USA-CPI-U",
                "observation_lag": "3M",
                "base_cpi": 315.5,
                "source_base_date": "2024-12-01",
                "source_base_cpi": 315.5,
                "day_count": "act_365f",
                "extrapolation": "extend",
                "method": "bootstrap",
                "interpolation": "linear",
            },
        ],
        "settings": {"use_parallel": False},
    },
    "market_data": rate_quotes + inflation_quotes + [{"kind": "inflation_fixings", **json.loads(fixings.to_json())}],
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print(result.to_dataframe().to_string(index=False))
print()

cal_curve = result.market.get_inflation_curve("USD-CPI")
print("Calibrated CPI path:")
for t in (1.0, 3.0, 5.0, 10.0):
    print(
        f"  t={t:>4.1f}y  cpi={cal_curve.cpi(t):.3f}"
        f"  annualized_inflation={cal_curve.inflation_rate(0.0, t):.6f}"
    )
